# Domain_Behaviour — how does each domain behave in this COSMoS release?

Re-derives, from the published COSMoS content, the behaviour that the Findings
consumers' scope decisions rest on, and reports where content and decisions disagree.
It reports only. It never changes a scope or a classification; those stay in
`SDTM_Domain_Metadata.xlsx`.

This is the March 2026 behavioural analysis
(`docs/archive/behavioural-analysis-2026-03/`) made reproducible: run it on every
COSMoS release and compare with the previous run.

**Inputs**
- `cosmos-graph/interim/COSMoS_Graph.xlsx` — DSS and BC sheets
- `consumer-bases/interim/DSS_View.xlsx` — `Measurement_Specs` (one row per DSS, variables pivoted)
- `sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx` — `Domains`,
  `Instrument_Domain_Rules`, `Consumer_Exclusions`

**Output** — `domain-behaviour/interim/Domain_Behaviour.xlsx`

| Sheet | Grain | Content |
|---|---|---|
| `Domain_Profile` | domain | Size, fan-out, primary axis, scale mix, variable shares; metadata class and consumer status |
| `Fanout_Axes` | BC with more than one DSS | Which axes distinguish its DSSs |
| `Scale_Units` | domain × published result-scale combination | DSSs with and without units; flags |
| `Exclusion_Claims` | claim in `Consumer_Exclusions` | Holds / does not hold, with the evidence |
| `Unclassified_Content` | domain | Domains with DSSs but no consumer class |
| `Key_Measures`, `Pins` | — | Measures for the release-to-release diff; input versions |

## 1. Setup

In [1]:
from pathlib import Path
from collections import Counter

import pandas as pd

REPO = Path.cwd().parent.parent  # notebooks/ -> domain-behaviour/ -> repo root
GRAPH = REPO / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'
DSS_VIEW = REPO / 'consumer-bases' / 'interim' / 'DSS_View.xlsx'
DOMAIN_META = REPO / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
OUT = REPO / 'domain-behaviour' / 'interim' / 'Domain_Behaviour.xlsx'

# Optional: path to a previous Domain_Behaviour.xlsx for the release diff in the final cell.
PRIOR_FILE = None

for path in [GRAPH, DSS_VIEW, DOMAIN_META]:
    if not path.exists():
        raise FileNotFoundError(f'{path} not found')
OUT.parent.mkdir(exist_ok=True)
print('repo:', REPO)

repo: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai


## 2. Load

In [2]:
dss = pd.read_excel(GRAPH, sheet_name='DSS', dtype=str).fillna('')
bc = pd.read_excel(GRAPH, sheet_name='BC', dtype=str).fillna('')
view = pd.read_excel(DSS_VIEW, sheet_name='Measurement_Specs', dtype=str).fillna('')
meta = pd.read_excel(DOMAIN_META, sheet_name='Domains', dtype=str).fillna('')
instr_rules = pd.read_excel(DOMAIN_META, sheet_name='Instrument_Domain_Rules', dtype=str).fillna('')
excl = pd.read_excel(DOMAIN_META, sheet_name='Consumer_Exclusions', dtype=str).fillna('')

if len(view) != len(dss):
    raise ValueError(f'DSS_View has {len(view)} DSS rows, COSMoS_Graph DSS sheet has {len(dss)} - rebuild consumer-bases first')

package_date = dss['package_date'].max()
print(f'COSMoS package_date (latest): {package_date}')
print(f'DSSs: {len(dss):,}  BCs with DSS: {dss["bc_id"].nunique():,}  domains: {dss["domain"].nunique()}')

COSMoS package_date (latest): 2026-07-14
DSSs: 1,475  BCs with DSS: 1,008  domains: 32


## 3. Axes

An axis is a variable whose value can differ between the DSSs of one BC. It is counted
when the assigned value differs. Where only the value list differs, the axis is reported
separately (`Axes_Value_List_Only`). Units, codelists,
LOINC codes and result values follow from the axes and are not counted as axes.
`scale` is read from units: some DSSs of the BC carry units, others do not.

In [3]:
AXIS_VARIABLES = {
    'specimen': ['SPEC'],
    'method': ['METHOD', 'ANMETH'],
    'binding agent': ['BDAGNT'],
    'test detail': ['TSTDTL'],
    'location': ['LOC', 'LAT', 'POS', 'DIR', 'PORTOT'],
    'category': ['CAT', 'SCAT'],
    'evaluator': ['EVAL', 'EVALID'],
}
AXIS_COLUMNS = {
    axis: [f'{v}{s}' for v in variables for s in ('_value', '_value_list') if f'{v}{s}' in view.columns]
    for axis, variables in AXIS_VARIABLES.items()
}
UNIT_COLUMNS = [c for c in view.columns if c.startswith('ORRESU_')]
view['has_units'] = view[UNIT_COLUMNS].ne('').any(axis=1)

for axis, cols in AXIS_COLUMNS.items():
    print(f'  {axis:14} {cols}')

  specimen       ['SPEC_value', 'SPEC_value_list']
  method         ['METHOD_value', 'METHOD_value_list', 'ANMETH_value', 'ANMETH_value_list']
  binding agent  ['BDAGNT_value']
  test detail    ['TSTDTL_value']
  location       ['LOC_value', 'LOC_value_list', 'LAT_value_list', 'POS_value_list', 'DIR_value_list']
  category       ['CAT_value', 'CAT_value_list', 'SCAT_value', 'SCAT_value_list']
  evaluator      ['EVAL_value', 'EVAL_value_list', 'EVALID_value_list']


## 4. Fan-out axes per BC

In [4]:
rows = []
for (domain, bc_id), g in view.groupby(['domain', 'bc_id']):
    if len(g) < 2:
        continue
    axes = [axis for axis, cols in AXIS_COLUMNS.items() if any(g[c].nunique() > 1 for c in cols if c.endswith('_value'))]
    list_only = [axis for axis, cols in AXIS_COLUMNS.items()
                 if axis not in axes and any(g[c].nunique() > 1 for c in cols if c.endswith('_value_list'))]
    if g['has_units'].nunique() > 1:
        axes.append('scale')
    rows.append({
        'domain': domain,
        'bc_id': bc_id,
        'bc_short_name': g['bc_short_name'].iloc[0],
        'DSS_Count': len(g),
        'Axes': '; '.join(axes) if axes else '(none)',
        'Axes_Value_List_Only': '; '.join(list_only),
        'DS_Codes': '; '.join(sorted(g['ds_id'])),
    })
fanout = pd.DataFrame(rows).sort_values(['domain', 'bc_id']).reset_index(drop=True)
print(f'BCs with more than one DSS: {len(fanout)}')

BCs with more than one DSS: 83


## 5. Domain profile

In [5]:
instrument_domains = set(instr_rules['Domain'])

def consumer_class(row):
    if row['Specimen_Based'] == 'Yes':
        return 'Specimen'
    if row['Measurement'] == 'Yes':
        return 'Measurement'
    if row['Domain'] in instrument_domains:
        return 'Instrument'
    return ''

meta['Consumer_Class'] = meta.apply(consumer_class, axis=1)
CONSUMER_OF = {'Specimen': 'Specimen_Findings', 'Measurement': 'Measurement_Findings', 'Instrument': 'Instrument_Findings'}
excluded = set(zip(excl['Consumer'], excl['Domain']))

def share(g, cols):
    cols = [c for c in cols if c in g.columns]
    return round(g[cols].ne('').any(axis=1).mean(), 2) if cols else 0.0

profile_rows = []
for domain, g in view.groupby('domain'):
    per_bc = g.groupby('bc_id').size()
    f = fanout[fanout['domain'] == domain]
    axis_counts = Counter(a for axes in f['Axes'] for a in axes.split('; '))
    top = max(axis_counts.values()) if axis_counts else 0
    primary = '; '.join(sorted(a for a, n in axis_counts.items() if n == top)) if axis_counts else ''
    scales = Counter(s for v in g['result_scales'] for s in v.split(';') if s)
    m = meta[meta['Domain'] == domain]
    cls = m['Consumer_Class'].iloc[0] if len(m) else ''
    if not len(m):
        status = 'not in domain metadata'
    elif not cls:
        status = 'no consumer class'
    elif (CONSUMER_OF[cls], domain) in excluded:
        status = f'excluded from {CONSUMER_OF[cls]}'
    else:
        status = f'in {CONSUMER_OF[cls]}'
    profile_rows.append({
        'domain': domain,
        'Observation_Class': m['Observation_Class'].iloc[0] if len(m) else '',
        'Consumer_Class': cls,
        'Consumer_Status': status,
        'BCs': per_bc.size,
        'DSSs': len(g),
        'BCs_Fanning_Out': int((per_bc > 1).sum()),
        'Max_Fanout': int(per_bc.max()),
        'Primary_Axis': primary,
        'Axes_Seen': '; '.join(f'{a} {n}' for a, n in sorted(axis_counts.items(), key=lambda kv: (-kv[1], kv[0]))),
        'Scale_Mix': '; '.join(f'{s} {n}' for s, n in sorted(scales.items(), key=lambda kv: (-kv[1], kv[0]))),
        'Share_With_Units': round(g['has_units'].mean(), 2),
        'Share_With_Specimen': share(g, AXIS_COLUMNS['specimen']),
        'Share_With_Method': share(g, AXIS_COLUMNS['method']),
        'Share_With_Location': share(g, AXIS_COLUMNS['location']),
    })
profile = pd.DataFrame(profile_rows)
with pd.option_context('display.width', 220, 'display.max_columns', None, 'display.max_rows', None):
    print(profile[['domain', 'Consumer_Status', 'BCs', 'DSSs', 'BCs_Fanning_Out', 'Max_Fanout', 'Primary_Axis']].to_string(index=False))

domain                 Consumer_Status  BCs  DSSs  BCs_Fanning_Out  Max_Fanout       Primary_Axis
    AE               no consumer class    1     2                1           2             (none)
    BE               no consumer class    7     7                0           1                   
    CM               no consumer class    1     4                1           4           category
    DD               no consumer class   13    13                0           1                   
    DM               no consumer class    5     5                0           1                   
    DS               no consumer class   20    20                0           1                   
    EC               no consumer class    1     2                1           2             (none)
    EG         in Measurement_Findings   33    33                0           1                   
    EX               no consumer class    1     1                0           1                   
    FA              

## 6. Scale and units

Grain: domain × the result-scale combination COSMoS publishes on the BC. Flags only for
single-scale combinations: Quantitative without units, and Nominal or Ordinal with units.

In [6]:
su = view.groupby(['domain', 'result_scales']).agg(DSSs=('ds_id', 'size'), With_Units=('has_units', 'sum')).reset_index()
su['Without_Units'] = su['DSSs'] - su['With_Units']

def su_flag(r):
    if r['result_scales'] == 'Quantitative' and r['Without_Units'] > 0:
        return 'Quantitative without units'
    if r['result_scales'] in ('Nominal', 'Ordinal') and r['With_Units'] > 0:
        return f'{r["result_scales"]} with units'
    return ''

su['Flag'] = su.apply(su_flag, axis=1)
scale_units = su.rename(columns={'result_scales': 'Result_Scales'})
print(scale_units[scale_units['Flag'] != ''].to_string(index=False))

domain Result_Scales  DSSs  With_Units  Without_Units                       Flag
    DM  Quantitative     1           0              1 Quantitative without units
    LB  Quantitative    76          72              4 Quantitative without units
    MB  Quantitative     2           1              1 Quantitative without units
    MI  Quantitative     1           0              1 Quantitative without units
    MK  Quantitative    25          21              4 Quantitative without units
    RE  Quantitative   125         110             15 Quantitative without units
    RP  Quantitative    37          10             27 Quantitative without units
    RS       Ordinal   124          19            105         Ordinal with units
    SC  Quantitative    10           2              8 Quantitative without units
    SR  Quantitative     6           5              1 Quantitative without units
    TS  Quantitative     8           0              8 Quantitative without units
    UR  Quantitative     5  

## 7. Exclusion claims

Each row of `Consumer_Exclusions` states one claim in machine form (`Check`):

- `no_fanout` — no BC in the domain has more than one DSS
- `no_axis` — no BC that fans out differs by any axis variable
- `not_axis:<axis>` — no BC that fans out differs by `<axis>`
- `axis_all:<axis>` — every BC that fans out differs by `<axis>`
- `not_primary:<axis>` — `<axis>` is not the primary axis (the axis most BCs that fan out differ by)

In [7]:
def evaluate(domain, check):
    f = fanout[fanout['domain'] == domain]
    n = len(f)
    if not (view['domain'] == domain).any():
        return 'no content', 'no DSSs in this release'
    if check == 'no_fanout':
        return ('yes' if n == 0 else 'no'), f'{n} BCs fan out'
    axis_sets = [set(a.split('; ')) - {'(none)'} for a in f['Axes']]
    seen = Counter(a for s in axis_sets for a in s)
    evidence = f'{n} BCs fan out; axes: ' + ('; '.join(f'{a} {c}' for a, c in sorted(seen.items(), key=lambda kv: (-kv[1], kv[0]))) or 'none')
    if check == 'no_axis':
        return ('yes' if not seen else 'no'), evidence
    kind, _, axis = check.partition(':')
    if axis not in AXIS_COLUMNS and axis != 'scale':
        raise ValueError(f'Unknown axis in check {check!r}')
    if kind == 'not_axis':
        return ('yes' if seen[axis] == 0 else 'no'), evidence
    if kind == 'axis_all':
        return ('yes' if n and seen[axis] == n else 'no'), evidence
    if kind == 'not_primary':
        top = max(seen.values()) if seen else 0
        return ('yes' if seen[axis] < top else 'no'), evidence
    raise ValueError(f'Unknown check {check!r}')

claims = excl.copy()
claims[['Holds', 'Evidence']] = claims.apply(lambda r: pd.Series(evaluate(r['Domain'], r['Check'])), axis=1)
claims = claims[['Consumer', 'Domain', 'Claim_ID', 'Claim', 'Check', 'Holds', 'Evidence', 'Reason']]
with pd.option_context('display.width', 220, 'display.max_colwidth', 90):
    print(claims[['Claim_ID', 'Claim', 'Holds', 'Evidence']].to_string(index=False))

Claim_ID                                           Claim Holds                                                                           Evidence
    IS-1 Every BC that fans out differs by binding agent   yes 6 BCs fan out; axes: binding agent 6; scale 4; test detail 2; method 1; specimen 1
    IS-2                Specimen is not the primary axis   yes 6 BCs fan out; axes: binding agent 6; scale 4; test detail 2; method 1; specimen 1
    GF-1   Every BC that fans out differs by test detail   yes                10 BCs fan out; axes: test detail 10; method 4; scale 4; specimen 1
    GF-2                Specimen is not the primary axis   yes                10 BCs fan out; axes: test detail 10; method 4; scale 4; specimen 1
    UR-1                     No BC has more than one DSS   yes                                                                      0 BCs fan out


## 8. Content without a consumer class

In [8]:
unclassified = profile[profile['Consumer_Status'].isin(['no consumer class', 'not in domain metadata'])].reset_index(drop=True)
print(', '.join(unclassified['domain']))

AE, BE, CM, DD, DM, DS, EC, EX, FA, IE, MH, PR, RP, SC, SR, SU, TR, TS, TU


## 9. Key measures and write

In [9]:
measures = [
    ('package_date', package_date),
    ('domains_with_dss', profile.shape[0]),
    ('dss', len(view)),
    ('bcs_with_dss', view['bc_id'].nunique()),
    ('bcs_fanning_out', len(fanout)),
    ('claims', len(claims)),
    ('claims_not_holding', int((claims['Holds'] == 'no').sum())),
    ('scale_unit_flags', int((scale_units['Flag'] != '').sum())),
    ('domains_with_dss_no_consumer_class', len(unclassified)),
]
for _, r in profile.iterrows():
    measures += [
        (f'dss:{r["domain"]}', r['DSSs']),
        (f'bcs_fanning_out:{r["domain"]}', r['BCs_Fanning_Out']),
        (f'primary_axis:{r["domain"]}', r['Primary_Axis']),
    ]
for _, r in claims.iterrows():
    measures.append((f'claim:{r["Claim_ID"]}', r['Holds']))
key_measures = pd.DataFrame(measures, columns=['measure', 'value']).astype({'value': str})

pins = pd.DataFrame([
    ('COSMoS package_date (latest in DSS sheet)', package_date),
    ('COSMoS_Graph.xlsx modified', pd.Timestamp(GRAPH.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('DSS_View.xlsx modified', pd.Timestamp(DSS_VIEW.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
    ('SDTM_Domain_Metadata.xlsx modified', pd.Timestamp(DOMAIN_META.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')),
], columns=['pin', 'value'])

with pd.ExcelWriter(OUT, engine='openpyxl') as writer:
    profile.to_excel(writer, sheet_name='Domain_Profile', index=False)
    fanout.to_excel(writer, sheet_name='Fanout_Axes', index=False)
    scale_units.to_excel(writer, sheet_name='Scale_Units', index=False)
    claims.to_excel(writer, sheet_name='Exclusion_Claims', index=False)
    unclassified.to_excel(writer, sheet_name='Unclassified_Content', index=False)
    key_measures.to_excel(writer, sheet_name='Key_Measures', index=False)
    pins.to_excel(writer, sheet_name='Pins', index=False)
print('written:', OUT)

written: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/domain-behaviour/interim/Domain_Behaviour.xlsx


## 10. Release diff (set `PRIOR_FILE` above)

In [10]:
if PRIOR_FILE and Path(PRIOR_FILE).exists():
    prior = pd.read_excel(PRIOR_FILE, sheet_name='Key_Measures', dtype=str)
    prev = dict(zip(prior['measure'], prior['value']))
    curr = dict(zip(key_measures['measure'], key_measures['value']))
    print(f'Prior: {PRIOR_FILE}')
    print('Added measures:  ', sorted(set(curr) - set(prev)))
    print('Dropped measures:', sorted(set(prev) - set(curr)))
    changed = [(m, prev[m], curr[m]) for m in sorted(set(prev) & set(curr)) if prev[m] != curr[m]]
    print('Value changes (measure: prior -> current):')
    for m, a, b in changed:
        print(f'  {m}: {a} -> {b}')
    if not changed:
        print('  (none)')
else:
    print('PRIOR_FILE not set or missing - skipping release diff.')

PRIOR_FILE not set or missing - skipping release diff.


## 11. Publish as HTML

Writes `docs/analyses/domain-behaviour.html`, served by GitHub Pages and linked from the
landing page. Regenerated on every run, so the page always shows the last run's release.

In [11]:
import html
from datetime import date

HTML_OUT = REPO / 'docs' / 'analyses' / 'domain-behaviour.html'
HTML_OUT.parent.mkdir(exist_ok=True)
GH = 'https://github.com/kerfors/cdisc-for-ai'
e = lambda s: html.escape(str(s))

def table(df, cols, headers, cls=''):
    head = ''.join(f'<th>{e(h)}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{v}</td>' for v in row) + '</tr>' for row in df[cols].itertuples(index=False))
    return f'<div class="scroll"><table class="{cls}"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

# Exclusion claims
c = claims.astype(str).copy()
c['Holds'] = c['Holds'].map(lambda h: f'<span class="{"ok" if h == "yes" else "no" if h == "no" else "na"}">{e(h)}</span>')
c['Claim'] = c.apply(lambda r: f'{e(r["Claim"])}<span class="desc">{e(r["Reason"])}</span>', axis=1)
for col in ['Claim_ID', 'Domain', 'Evidence']:
    c[col] = c[col].map(e)
claims_html = table(c, ['Claim_ID', 'Domain', 'Claim', 'Holds', 'Evidence'], ['Claim', 'Domain', 'Statement and reason', 'Holds', 'Evidence'])

# Domains: those with a consumer class or fan-out in the table, the rest folded
ORDER = ['in Specimen_Findings', 'excluded from Specimen_Findings', 'in Measurement_Findings',
         'in Instrument_Findings', 'no consumer class', 'not in domain metadata']
p = profile.copy()
p['_o'] = p['Consumer_Status'].map(lambda s: ORDER.index(s) if s in ORDER else 99)
p = p.sort_values(['_o', 'domain'])
main = p[(p['Consumer_Class'] != '') | (p['BCs_Fanning_Out'] > 0)].astype(str)
rest = p[(p['Consumer_Class'] == '') & (p['BCs_Fanning_Out'] == 0)].astype(str)
for d in (main, rest):
    for col in d.columns:
        d[col] = d[col].map(e)
PROFILE_COLS = ['domain', 'Consumer_Status', 'BCs', 'DSSs', 'BCs_Fanning_Out', 'Max_Fanout', 'Primary_Axis', 'Scale_Mix']
PROFILE_HEAD = ['Domain', 'Consumer', 'BCs', 'DSSs', 'BCs that fan out', 'Max fan-out', 'Primary axis', 'Result scales (BC)']
profile_html = table(main, PROFILE_COLS, PROFILE_HEAD, 'num')
if len(rest):
    profile_html += (f'<details><summary>{len(rest)} more domains with no consumer class and no fan-out: '
                     f'{e(", ".join(rest["domain"]))}</summary>' + table(rest, PROFILE_COLS, PROFILE_HEAD, 'num') + '</details>')

# Fan-out axes per domain, collapsible
blocks = []
for dom, g in fanout.groupby('domain'):
    g = g.astype(str).copy()
    for col in g.columns:
        g[col] = g[col].map(e)
    blocks.append(f'<details><summary><strong>{e(dom)}</strong> · {len(g)} BC{"s" if len(g) != 1 else ""} fan{"" if len(g) != 1 else "s"} out</summary>'
                  + table(g, ['bc_id', 'bc_short_name', 'DSS_Count', 'Axes', 'Axes_Value_List_Only'], ['BC', 'Name', 'DSSs', 'Axes that differ', 'Only the value list differs']) + '</details>')
fanout_html = '<div class="grid">' + '\n'.join(blocks) + '</div>'

su = scale_units[scale_units['Flag'] != ''].astype(str).copy()
for col in su.columns:
    su[col] = su[col].map(e)
su_html = table(su, ['domain', 'Result_Scales', 'DSSs', 'With_Units', 'Without_Units', 'Flag'],
                ['Domain', 'Result scale (BC)', 'DSSs', 'With units', 'Without units', 'Flag'], 'num3')

page = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Domain behaviour</title>
<style>
    :root {{ --text:#1a1a1a; --muted:#555; --rule:#e5e5e5; --link:#0b5fa5; --bg:#fdfdfd; --ok:#2e7d32; --no:#b3261e; }}
    html {{ box-sizing:border-box; }} *, *::before, *::after {{ box-sizing:inherit; }}
    body {{ font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif; color:var(--text); background:var(--bg);
           line-height:1.55; max-width:60rem; margin:3rem auto; padding:0 1rem; font-size:16px; }}
    h1 {{ font-size:1.75rem; font-weight:600; margin:0 0 .5rem; letter-spacing:-.01em; }}
    h2 {{ font-size:1.15rem; font-weight:600; margin:2.5rem 0 .75rem; padding-bottom:.35rem; border-bottom:1px solid var(--rule); }}
    p {{ margin:.75rem 0; max-width:42rem; }}
    a {{ color:var(--link); text-decoration:none; }} a:hover {{ text-decoration:underline; }}
    code {{ font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace; font-size:.9em; background:#f3f3f3; padding:.1em .35em; border-radius:3px; }}
    .lead {{ color:var(--muted); font-size:1.05rem; margin:-.25rem 0 1.5rem; line-height:1.4; }}
    .pins {{ color:var(--muted); font-size:.92em; }}
    .scroll {{ overflow-x:auto; margin:.5rem 0 1rem; }}
    table {{ border-collapse:collapse; width:100%; font-size:.9em; }}
    th, td {{ text-align:left; vertical-align:top; padding:.4rem .6rem; border-bottom:1px solid var(--rule); }}
    th {{ font-weight:600; white-space:nowrap; }}
    table.num td:nth-child(n+3):nth-child(-n+6), table.num3 td:nth-child(n+3):nth-child(-n+5) {{ text-align:right; font-variant-numeric:tabular-nums; }}
    .desc {{ display:block; color:var(--muted); font-size:.92em; margin-top:.15rem; }}
    .ok {{ color:var(--ok); font-weight:600; }} .no {{ color:var(--no); font-weight:600; }} .na {{ color:var(--muted); }}
    details {{ margin:.4rem 0; }} summary {{ cursor:pointer; padding:.25rem 0; }}
    .grid {{ display:grid; grid-template-columns:repeat(auto-fill, minmax(17rem, 1fr)); column-gap:1.5rem; }}
    .grid details[open] {{ grid-column:1 / -1; }}
    footer {{ margin-top:3rem; padding-top:1rem; border-top:1px solid var(--rule); color:var(--muted); font-size:.9em; }}
</style>
</head>
<body>
<p class="pins"><a href="../index.html">CDISC for AI</a> · Analyses</p>
<h1>Domain behaviour</h1>
<p class="lead">How each SDTM domain behaves in the published COSMoS content: how Biomedical Concepts fan out into Dataset Specializations, and which variables make the difference.</p>

<p>One schema serves every COSMoS domain, but the relationship between a Biomedical Concept (BC) and its Dataset Specializations (DSSs) behaves differently from domain to domain. This page re-derives that behaviour from each COSMoS release. An <em>axis</em> is a variable whose value differs between the DSSs of one BC: specimen, method, binding agent, test detail, location, category, evaluator. An axis is counted when the assigned value differs; where only the value list differs, it is shown in a separate column. <em>Scale</em> is read from units: some DSSs of the BC carry units, others do not.</p>
<p>The page reports; it does not decide. Which domains each Findings consumer covers is recorded in <a href="{GH}/blob/main/sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx"><code>SDTM_Domain_Metadata.xlsx</code></a>, and the reasons for leaving a domain out are stated there as claims. The first table tests those claims against this release.</p>
<p class="pins">COSMoS package {e(package_date)} · generated {date.today().isoformat()} · data: <a href="{GH}/raw/main/domain-behaviour/interim/Domain_Behaviour.xlsx"><code>Domain_Behaviour.xlsx</code></a> · notebook: <a href="{GH}/blob/main/domain-behaviour/notebooks/Domain_Behaviour.ipynb"><code>Domain_Behaviour.ipynb</code></a></p>

<h2>Exclusion claims</h2>
<p>Each reason for leaving a domain out of a consumer, stated as something the published content must show.</p>
{claims_html}

<h2>Domains</h2>
<p>One row per domain with Dataset Specializations. The primary axis is the axis most BCs that fan out differ by.</p>
{profile_html}

<h2>Where the DSSs of one BC differ</h2>
{fanout_html}

<h2>Scale and units</h2>
<details><summary>{len(su)} result-scale combinations where units and scale do not line up</summary>
<p>Quantitative without units, or Nominal and Ordinal with units. Reported, not judged — an ordinal score item can collect a raw value with units.</p>
{su_html}
</details>

<footer>
Part of <a href="../index.html">CDISC for AI</a>. The March 2026 analysis this page continues is archived in <a href="{GH}/tree/main/docs/archive/behavioural-analysis-2026-03"><code>docs/archive/behavioural-analysis-2026-03/</code></a>. Exploratory work, not an official CDISC product.
</footer>
</body>
</html>
'''
HTML_OUT.write_text(page, encoding='utf-8')
print('written:', HTML_OUT)

written: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/docs/analyses/domain-behaviour.html
